# Point clouds

These are the array moves behind lidar preprocessing: bounds, nearest neighbors, and voxel centroids. Shapes stay `(N, 3)` unless a problem says otherwise.

For a large cloud, say what you would do in production (chunk the broadcast, or `scipy.spatial.cKDTree`) after the vectorized version works.


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)


## 1. Bounds and unit-sphere normalization

Return the axis-aligned minimum, the axis-aligned maximum, and a normalized copy of the cloud.

Normalization subtracts the centroid and divides by the largest distance from that centroid, so the result sits in the unit ball and touches the sphere. If every point is the same, return the centered zeros and do not divide.

Hand check: `(0,0,0)`, `(2,0,0)`, `(0,2,0)` has min `(0,0,0)`, max `(2,2,0)`, a centered result, and max norm 1.


In [ ]:
import numpy as np

def bounds_and_unit_sphere(points):
    """
    AABB min/max and a copy centered at the origin, scaled into the unit ball.

    The scale is the max distance from the centroid, so the result touches the
    unit sphere. A degenerate cloud (all points equal) is returned centered.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def bounds_and_unit_sphere(points):
    """AABB min/max and a copy centered at the origin, scaled into the unit ball.

    The scale is the max distance from the centroid, so the result touches the
    unit sphere. A degenerate cloud (all points equal) is returned centered.
    """
    points = np.asarray(points, dtype=float)
    mn = points.min(axis=0)
    mx = points.max(axis=0)
    center = points.mean(axis=0)
    centered = points - center
    scale = np.linalg.norm(centered, axis=1).max()
    if scale < 1e-12:
        normalized = centered
    else:
        normalized = centered / scale
    return mn, mx, normalized

def check_bounds_and_unit_sphere():
    pts = np.array([[0.0, 0.0, 0.0], [2.0, 0.0, 0.0], [0.0, 2.0, 0.0]])
    mn, mx, normed = bounds_and_unit_sphere(pts)
    assert np.allclose(mn, [0, 0, 0])
    assert np.allclose(mx, [2, 2, 0])
    assert np.allclose(normed.mean(axis=0), 0, atol=1e-8)
    assert np.allclose(np.linalg.norm(normed, axis=1).max(), 1.0)
    flat = bounds_and_unit_sphere(np.ones((4, 3)))
    assert np.allclose(flat[2], 0, atol=1e-8)
    print("passed")

check_bounds_and_unit_sphere()


## 2. Nearest neighbors

For every point in `A`, find the closest point in `B`. Return indices into `B` and Euclidean distances.

Do it with broadcasting, not a Python loop over points. Then say the memory cost out loud.

Hand check: `A = [(0,0,0), (1,0,0)]` and `B = [(0.1,0,0), (5,0,0), (0.9,0.1,0)]` matches indices `[0, 2]`.

<details><summary>Hint</summary>

`A[:, None, :] - B[None, :, :]` has shape `(N, M, D)`. `argmin` over axis 1. `N = M = 10000` is about 800 MB of float64 distances. Chunk `A`, or use a KD-tree, past that.

</details>


In [ ]:
import numpy as np

def nearest_neighbors(A, B):
    """
    Brute-force nearest neighbor in B for every point of A.

    A: (N, D), B: (M, D)
    returns idx (N,) into B, and Euclidean distances (N,).
    Memory is O(N*M). Chunk A, or use a KD-tree, when that product is large.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def nearest_neighbors(A, B):
    """Brute-force nearest neighbor in B for every point of A.

    A: (N, D), B: (M, D)
    returns idx (N,) into B, and Euclidean distances (N,).
    Memory is O(N*M). Chunk A, or use a KD-tree, when that product is large.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    if len(B) == 0:
        raise ValueError("B is empty")
    delta = A[:, None, :] - B[None, :, :]
    dist = np.linalg.norm(delta, axis=-1)
    idx = np.argmin(dist, axis=1)
    return idx, dist[np.arange(len(A)), idx]

def check_nearest_neighbors():
    A = np.array([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]])
    B = np.array([[0.1, 0.0, 0.0], [5.0, 0.0, 0.0], [0.9, 0.1, 0.0]])
    idx, dist = nearest_neighbors(A, B)
    assert np.array_equal(idx, [0, 2])
    assert np.allclose(dist, [0.1, np.sqrt(0.02)])
    print("passed")

check_nearest_neighbors()


## 3. Chamfer distance

Symmetric Chamfer distance, defined here as

`mean over A of (distance to nearest B) + mean over B of (distance to nearest A)`.

Some papers divide by two, and some sum squared distances. Say which one you implemented.

Hand check: `A = [(0,0,0), (10,0,0)]` and `B = [(0,0,0)]` gives `5`, because one side contributes 0 and the other contributes the average of 0 and 10. Two unit squares sitting one unit apart, `A = [(0,0,0),(1,0,0)]` and `B = [(0,1,0),(1,1,0)]`, give `2`.

The reference cell includes a nearest-neighbor helper so it runs alone. Your attempt can define its own helper in the same cell.


In [ ]:
import numpy as np

def chamfer_distance(A, B):
    """
    Sum of the two directed mean nearest-neighbor distances.

    mean_a min_b ||a-b|| + mean_b min_a ||b-a||.
    Some papers divide by 2. Say which definition you are using.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

# Included so this cell runs by itself, including helpers from earlier problems.

def nearest_neighbors(A, B):
    """Brute-force nearest neighbor in B for every point of A.

    A: (N, D), B: (M, D)
    returns idx (N,) into B, and Euclidean distances (N,).
    Memory is O(N*M). Chunk A, or use a KD-tree, when that product is large.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    if len(B) == 0:
        raise ValueError("B is empty")
    delta = A[:, None, :] - B[None, :, :]
    dist = np.linalg.norm(delta, axis=-1)
    idx = np.argmin(dist, axis=1)
    return idx, dist[np.arange(len(A)), idx]

def chamfer_distance(A, B):
    """Sum of the two directed mean nearest-neighbor distances.

    mean_a min_b ||a-b|| + mean_b min_a ||b-a||.
    Some papers divide by 2. Say which definition you are using.
    """
    _, d_ab = nearest_neighbors(A, B)
    _, d_ba = nearest_neighbors(B, A)
    return float(d_ab.mean() + d_ba.mean())

def check_chamfer_distance():
    A = np.array([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]])
    B = np.array([[0.0, 1.0, 0.0], [1.0, 1.0, 0.0]])
    assert np.allclose(chamfer_distance(A, B), 2.0)
    A = np.array([[0.0, 0.0, 0.0], [10.0, 0.0, 0.0]])
    B = np.array([[0.0, 0.0, 0.0]])
    assert np.allclose(chamfer_distance(A, B), 5.0)
    assert np.allclose(chamfer_distance([[0.0, 0.0, 0.0]], [[3.0, 4.0, 0.0]]), 10.0)
    print("passed")

check_chamfer_distance()


## 4. Voxel downsample

Replace each occupied voxel by the centroid of the points inside it. `voxel` is the cubic edge length. Return shape `(V, 3)`. Order does not matter.

This has to be correct for negative coordinates. A point at `(-0.1, 0, 0)` with `voxel = 1` is not in the same cell as `(0.1, 0, 0)`.

Hand check those two points, plus a second point in the positive cell, plus `(1.2, 0, 0)`. You should get three centroids, and the positive cell near the origin should be the **mean** of its points, not the last point written.

<details><summary>Hint</summary>

`np.floor(points / voxel).astype(np.int64)` bins negative numbers toward -∞. Casting with `.astype(int)` truncates toward 0 and merges `-0.1` with `+0.1`. Accumulate with `np.add.at`. The statement `sums[inverse] += points` silently drops duplicate indices.

</details>


In [ ]:
import numpy as np

def voxel_downsample(points, voxel):
    """
    One centroid per occupied voxel of cubic size `voxel`.

    points: (N, 3). Returns (V, 3). Empty input returns shape (0, 3).
    Row order does not matter. The result must be correct for negative coordinates.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import numpy as np

def voxel_downsample(points, voxel):
    """One centroid per occupied voxel of cubic size `voxel`.

    points: (N, 3). Returns (V, 3). Empty input returns shape (0, 3).
    Row order does not matter. The result must be correct for negative coordinates.
    """
    points = np.asarray(points, dtype=float)
    if len(points) == 0:
        return np.zeros((0, points.shape[1]))
    # floor bins toward -inf. astype(int) truncates toward 0 and merges -0.1 with +0.1.
    keys = np.floor(points / voxel).astype(np.int64)
    _, inverse, counts = np.unique(keys, axis=0, return_inverse=True, return_counts=True)
    sums = np.zeros((len(counts), points.shape[1]))
    # sums[inverse] += points drops duplicate indices. add.at accumulates them.
    np.add.at(sums, inverse, points)
    return sums / counts[:, None]

def check_voxel_downsample():
    pts = np.array(
        [
            [0.1, 0.1, 0.1],
            [0.2, 0.2, 0.4],
            [-0.1, 0.0, 0.0],
            [1.2, 0.0, 0.0],
        ]
    )
    got = voxel_downsample(pts, 1.0)
    expected = np.array(
        [
            [-0.1, 0.0, 0.0],
            [0.15, 0.15, 0.25],
            [1.2, 0.0, 0.0],
        ]
    )
    order = np.lexsort(np.round(got, 8).T[::-1])
    order_e = np.lexsort(np.round(expected, 8).T[::-1])
    assert np.allclose(got[order], expected[order_e], atol=1e-8)
    # Truncation toward zero would put -0.1 in the same bin as +0.1.
    assert len(got) == 3
    print("passed")

check_voxel_downsample()
